# K-Radar 前 50 帧：A dense unfolded Doppler RA + QC

参考 `visualize_kradar_dra.ipynb` 的原生 RA 绘图方式。四幅图均以 **azimuth 为横轴、range 为纵轴**，底层矩阵保持 `[256, 107]`，不重采样到 BEV。

- **A / dense**：`[Power, D_dense]`，所有 cell 均有速度候选。
- **QC / reliable**：同一个 dense 速度场，只显示可靠 mask 内的值；灰色为弃权。B 的 mask 不属于 VLA 输入。
- 匹配来源单独显示；静态位姿先验共识是 QC 信息，不代表独立真值。

Power 使用全 50 帧相同的 −60～0 dB 色阶，Doppler 使用相同的 −35～35 m/s 色阶。dense 速度是位姿辅助匹配的候选，不是 oracle 标签。

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from IPython.display import display

ROOT = Path('/storage/data/metaiot_data/lxin/RadarVLA/KRadarRelated')
DIAG = ROOT / 'diagnostics' / 'dense_abstain_unfolding_v1'
CACHE = Path('/storage/data/metaiot_data/lxin/KRadar/visualization_cache/seq1_first50')

frames = json.loads((DIAG / 'frames.json').read_text())
stems = [row['radar_stem'] for row in frames]
dense = np.load(DIAG / 'version_a_50x2x256x107.npy', mmap_mode='r')
with np.load(CACHE / 'radar_axes.npz') as axes:
    range_m = axes['range_m'].copy()
    azimuth_deg = axes['azimuth_deg'].copy()
assert len(stems) == 50 and len(set(stems)) == 50
assert dense.shape == (50, 2, 256, 107)
assert range_m.shape == (256,) and azimuth_deg.shape == (107,)
assert np.isfinite(dense).all()
GLOBAL_POWER_PEAK = float(np.max(dense[:, 0]))
assert GLOBAL_POWER_PEAK > 0
DR = float(np.median(np.diff(range_m)))
DA = float(np.median(np.diff(azimuth_deg)))
RA_EXTENT = [azimuth_deg[0] - DA/2, azimuth_deg[-1] + DA/2,
             range_m[0] - DR/2, range_m[-1] + DR/2]
print(f'{len(stems)} frames | {dense.shape} | first={stems[0]}, last={stems[-1]}')

In [ ]:
SOURCE_NAMES = ['two-sided', 'one-sided', 'neighbor', 'previous', 'k=0']
SOURCE_CMAP = ListedColormap(['#3087c4', '#31a8a3', '#e0b33b', '#c77345', '#777777'])
SOURCE_NORM = BoundaryNorm(np.arange(.5, 6.5), SOURCE_CMAP.N)
VELOCITY_CMAP = plt.get_cmap('coolwarm').copy()
VELOCITY_CMAP.set_bad('#aaaaaa')

def show_frame(index=25):
    """Show native RA Power, dense speed, reliable speed, and decision source."""
    index = int(index)
    if not 0 <= index < len(stems):
        raise IndexError(index)
    stem = stems[index]
    power, dense_speed = dense[index]
    with np.load(DIAG / 'version_b_qc' / f'{stem}.npz') as qc:
        reliable_speed = qc['doppler_reliable_ra_mps'].copy()
        reliable_mask = qc['reliable_mask'].copy()
        source_code = qc['source_code'].copy()
        static_consensus = qc['static_consensus_mask'].copy()
    assert power.shape == dense_speed.shape == reliable_speed.shape == source_code.shape == (256, 107)
    assert np.array_equal(reliable_speed[reliable_mask], dense_speed[reliable_mask])
    assert np.isnan(reliable_speed[~reliable_mask]).all()
    power_db = 10 * np.log10(np.maximum(power, GLOBAL_POWER_PEAK * 1e-12) / GLOBAL_POWER_PEAK)

    fig, ax = plt.subplots(2, 2, figsize=(15, 11), constrained_layout=True)
    kw = dict(origin='lower', aspect='auto', extent=RA_EXTENT, interpolation='nearest')
    p = ax[0, 0].imshow(power_db, cmap='magma', vmin=-60, vmax=0, **kw)
    d = ax[0, 1].imshow(dense_speed, cmap=VELOCITY_CMAP, vmin=-35, vmax=35, **kw)
    b = ax[1, 0].imshow(reliable_speed, cmap=VELOCITY_CMAP, vmin=-35, vmax=35, **kw)
    s = ax[1, 1].imshow(source_code, cmap=SOURCE_CMAP, norm=SOURCE_NORM, **kw)
    titles = ('Power · original RA', 'A · dense unfolded Doppler candidate',
              f'QC · reliable Doppler (gray = abstain; {reliable_mask.mean():.1%} valid)',
              'Dense k decision source')
    for axis, title in zip(ax.flat, titles):
        axis.set(title=title, xlabel='Azimuth (deg)', ylabel='Range (m)')
    fig.colorbar(p, ax=ax[0, 0], label='dB relative to 50-frame maximum')
    fig.colorbar(d, ax=ax[0, 1], label='m/s (+ receding)')
    fig.colorbar(b, ax=ax[1, 0], label='m/s (+ receding)')
    cb = fig.colorbar(s, ax=ax[1, 1], ticks=range(1, 6))
    cb.ax.set_yticklabels(SOURCE_NAMES)
    fig.suptitle(f'K-Radar seq1 | {index+1}/50 | {stem} | static consensus {static_consensus.mean():.1%}')
    return fig

fig = show_frame(25)  # 首次执行先显示第 26 帧；下方滑块可切换
plt.show()

In [ ]:
import ipywidgets as widgets

slider = widgets.IntSlider(value=25, min=0, max=49, step=1,
                           description='Frame', continuous_update=False)
view = widgets.Output()

def refresh(change=None):
    with view:
        view.clear_output(wait=True)
        fig = show_frame(slider.value)
        display(fig)
        plt.close(fig)

slider.observe(refresh, names='value')
display(slider, view)
refresh()

## 文件与解释

- `version_a_50x2x256x107.npy`：可评估的两通道 dense 候选。Power 为原缓存值。
- `version_b_qc/<stem>.npz`：`doppler_reliable_ra_mps`、`reliable_mask`、`static_consensus_mask`、`source_code`、`k_dense`。
- `results.json`、`summary.json`、`README.md`：覆盖率、折叠阶数与静态伪参考诊断。

首帧和尾帧只能单侧匹配，所以可靠图全灰；dense 图仍有回退后的有限速度。低功率区域与动态目标没有独立阶数真值。